# 03 — Baselines (Phase 1)

Three baselines, per the brief, each logged to `experiments/runs.csv` via
the shared `baselines.log_run` helper (replaces the inline CSV-writing
that was previously copy-pasted per baseline):

1. **Persistence** — tomorrow's min temp = today's min temp.
2. **Physical/threshold model** — linear regression restricted to
   physically-motivated inputs (Tmin, dew-point depression, cloud cover) —
   this project's interpretable stand-in for a literature-style
   "Franklin/Young"-style empirical formula (see `baselines.py` docstring
   for why we fit rather than hardcode coefficients).
3. **Logistic regression** — plain classifier on the same feature set,
   class-weighted, predicting the label directly. This is the
   *documented ablation* against this project's primary
   regression-then-threshold approach, not the headline model — expect
   (and report) a precision/recall trade-off here, per Omazić et al.
   (2024)'s own class-weighted-XGBoost-vs-threshold finding.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

from data_loader import load_tier1_raw, get_city_frame
from features import add_dew_point_features
from label_rules import build_labels_for_horizon
from baselines import persistence_predict, PhysicalThresholdBaseline, LogisticRegressionBaseline, log_run
from evaluate import get_continuous_metrics, get_classification_metrics

## Build the same labeled Basel dataset as `02_label_definition.ipynb`

(Re-built here rather than re-loaded from the interim CSV, so this
notebook stays runnable on its own — but the logic is identical, since
both go through `label_rules.build_labels_for_horizon`.)

In [2]:
CITY = 'BASEL'
HORIZON_DAYS = 1
DATASET_TAG = f"{CITY.title()}_Tier1"

raw = load_tier1_raw('../data/raw/weather_prediction_dataset.csv')
city_df = get_city_frame(raw, CITY)
city_df = add_dew_point_features(city_df, temp_col='temp_min', rh_percent_col='humidity_pct')
labeled = build_labels_for_horizon(city_df, horizon=HORIZON_DAYS)
labeled.shape

(3653, 17)

## Time-based train/test split

This is a time series — a random split would leak future information into
training via autocorrelated neighboring days. Using the last 20% of rows
(chronologically) as a held-out test set instead.

In [3]:
split_idx = int(len(labeled) * 0.8)
train, test = labeled.iloc[:split_idx].copy(), labeled.iloc[split_idx:].copy()
print(f"Train: {len(train)} rows ({train['DATE'].min().date()} to {train['DATE'].max().date()})")
print(f"Test:  {len(test)} rows ({test['DATE'].min().date()} to {test['DATE'].max().date()})")

FROST_LABEL_COL = 'frost_label_broad'  # see 02_label_definition.ipynb for why broad is primary
CLASSIFICATION_THRESHOLD = 3.0

Train: 2922 rows (2000-01-01 to 2007-12-31)
Test:  731 rows (2008-01-01 to 2009-12-31)


## Baseline 1 — Persistence

In [4]:
pred_persistence = persistence_predict(test['temp_min'])

cont_metrics = get_continuous_metrics(test['target_min_temp'], pred_persistence)
pred_label = (pred_persistence <= CLASSIFICATION_THRESHOLD).astype(int)
class_metrics = get_classification_metrics(test[FROST_LABEL_COL], pred_label)

print("--- Persistence ---")
print(f"RMSE: {cont_metrics['rmse']:.2f}C  MAE: {cont_metrics['mae']:.2f}C")
print(f"Precision: {class_metrics['precision']:.2f}  Recall: {class_metrics['recall']:.2f}  "
      f"F1: {class_metrics['f1']:.2f}  Peirce: {class_metrics['peirce_skill']:.2f}")

log_run('../experiments/runs.csv', model='Persistence', dataset=DATASET_TAG,
        metrics={**cont_metrics, **class_metrics})

--- Persistence ---
RMSE: 2.43C  MAE: 1.87C
Precision: 0.84  Recall: 0.84  F1: 0.84  Peirce: 0.77


## Baseline 2 — Physical/threshold model

Restricted to physically-motivated inputs only (see `baselines.py` for
which columns it selects and why); predicts continuous Tmin, then applies
the same threshold as the label.

In [5]:
physical_model = PhysicalThresholdBaseline().fit(train, train['target_min_temp'])
pred_physical = physical_model.predict(test)

cont_metrics_p = get_continuous_metrics(test['target_min_temp'], pred_physical)
pred_label_p = (pred_physical <= CLASSIFICATION_THRESHOLD).astype(int)
class_metrics_p = get_classification_metrics(test[FROST_LABEL_COL], pred_label_p)

print("--- Physical/threshold baseline ---")
print(f"Features used: {physical_model.feature_columns_}")
print(f"RMSE: {cont_metrics_p['rmse']:.2f}C  MAE: {cont_metrics_p['mae']:.2f}C")
print(f"Precision: {class_metrics_p['precision']:.2f}  Recall: {class_metrics_p['recall']:.2f}  "
      f"F1: {class_metrics_p['f1']:.2f}  Peirce: {class_metrics_p['peirce_skill']:.2f}")

log_run('../experiments/runs.csv', model='PhysicalThreshold',
        params=str(physical_model.feature_columns_), dataset=DATASET_TAG,
        metrics={**cont_metrics_p, **class_metrics_p})

--- Physical/threshold baseline ---
Features used: ['temp_min', 'dew_point_depression_c', 'cloud_cover']
RMSE: 2.35C  MAE: 1.81C
Precision: 0.86  Recall: 0.85  F1: 0.85  Peirce: 0.78


## Baseline 3 — Logistic regression (documented ablation, not the primary approach)

Direct classifier on the label, class-weighted to counter frost being
rare. Compare its **recall** against the two regression-then-threshold
baselines above — per the brief's core design decision, a direct
classifier is expected to trade recall for precision relative to
threshold-on-continuous-temperature, and that trade-off (not a pipeline
bug) is exactly what should be reported here.

In [6]:
FEATURE_COLS = ['temp_min', 'dew_point_depression_c', 'cloud_cover']

logreg = LogisticRegressionBaseline().fit(train[FEATURE_COLS], train[FROST_LABEL_COL])
pred_logreg = logreg.predict(test[FEATURE_COLS])

class_metrics_lr = get_classification_metrics(test[FROST_LABEL_COL], pred_logreg)

print("--- Logistic regression (class-weighted, direct classifier) ---")
print(f"Features used: {FEATURE_COLS}")
print(f"Precision: {class_metrics_lr['precision']:.2f}  Recall: {class_metrics_lr['recall']:.2f}  "
      f"F1: {class_metrics_lr['f1']:.2f}  Peirce: {class_metrics_lr['peirce_skill']:.2f}")

log_run('../experiments/runs.csv', model='LogisticRegression_ClassWeighted',
        params=str(FEATURE_COLS), dataset=DATASET_TAG, metrics=class_metrics_lr)

--- Logistic regression (class-weighted, direct classifier) ---
Features used: ['temp_min', 'dew_point_depression_c', 'cloud_cover']
Precision: 0.81  Recall: 0.95  F1: 0.87  Peirce: 0.85


## Summary

`experiments/runs.csv` now has one row per baseline (plus whatever earlier
runs are already in there — the log is append-only by design, so re-running
this notebook adds new rows rather than overwriting history). Compare
Persistence vs. Physical/threshold vs. Logistic-regression recall here
directly against each other, and against the Phase 3 models once they
exist, in `reports/comparison_table.md`.

**Next step (Phase 2, `04_feature_engineering.ipynb`):** lag features,
rolling stats, and rate-of-change from `features.py`, using
`build_feature_set()` so training and (eventually) Repo B's edge pipeline
stay in sync.